In [5]:
pip install datasets pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Python\pythoncore-3.11-64\python.exe -m pip install --upgrade pip


In [8]:
%pip install nltk scikit-learn gensim joblib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
     ---------------------------------------- 0.0/41.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.6 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.6 kB ? eta -:--:--
     -------------------------------------- 41.6/41.6 kB 401.7 kB/s eta 0:00:00
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   - -------------------------------------- 0.1/1.7 MB 3.8 MB/s eta 0:00:01
   ---- ----------------------------------- 0.2/1.7 MB 3.5 MB/s eta 0:00:01
   ------- -------------------------------- 0.3/1.7 MB 2.6 MB/s eta 0:00:01
   ---------- ----------------------------- 0.4/1.7 MB 2.7 MB/s eta 0:00:01
   ------------- -------------------------- 0.6/1.7 MB 2.6 MB/s eta 0:00:01
   ---------------- ----------------------- 0.7/1.7

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Python\pythoncore-3.11-64\python.exe -m pip install --upgrade pip


In [11]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import gensim.downloader as api
from gensim.models import Word2Vec, FastText



In [12]:
# 1. Download required NLTK data (quietly)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# ==========================================
# DATA LOADING & PREPROCESSING
# ==========================================
print("Loading BBC News dataset from Hugging Face...")
dataset = load_dataset("SetFit/bbc-news")
df = pd.DataFrame(dataset['train'])
# Note: 'label_text' contains the category string (e.g., 'tech', 'business')
# 'text' contains the article body

print("Preprocessing text...")
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase, remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    # Tokenize, remove stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing
df['clean_text'] = df['text'].apply(preprocess_text)




Loading BBC News dataset from Hugging Face...
Preprocessing text...


In [13]:
# ==========================================
# LAB 1: TRADITIONAL TEXT REPRESENTATION
# ==========================================
print("\n--- Lab 1: Feature Extraction ---")

# 1. One-Hot Encoding 
# (max_features used to prevent RAM overflow on large corpora)
one_hot_vectorizer = CountVectorizer(binary=True, max_features=5000)
X_onehot = one_hot_vectorizer.fit_transform(df['clean_text'])
print(f"One-Hot Encoding Shape: {X_onehot.shape}")

# 2. Bag-of-Words (BoW)
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(df['clean_text'])
print(f"Bag-of-Words Shape: {X_bow.shape}")

# 3. TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])
print(f"TF-IDF Shape: {X_tfidf.shape}")





--- Lab 1: Feature Extraction ---
One-Hot Encoding Shape: (1225, 5000)
Bag-of-Words Shape: (1225, 5000)
TF-IDF Shape: (1225, 5000)


In [16]:
# ==========================================
# LAB 2: WORD EMBEDDINGS & CLASSIFICATION (UPDATED)
# ==========================================
print("\n--- Lab 2: Word Embeddings & Classification ---")

# Split Data
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'], df['label_text'], test_size=0.2, random_state=42
)

# Tokenize for Gensim models
train_tokens = [text.split() for text in X_train_text]
test_tokens = [text.split() for text in X_test_text]

# 1. Train Word2Vec & FastText on our corpus
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, workers=4)

print("Training FastText model...")
ft_model = FastText(sentences=train_tokens, vector_size=100, window=5, min_count=2, workers=4)

# 2. Load pre-trained GloVe
print("Loading pre-trained GloVe model (this downloads ~100MB on first run)...")
glove_model = api.load("glove-wiki-gigaword-100")

# Function to get average document vector
def get_doc_vector(tokens, model, vector_size=100):
    if hasattr(model, 'wv'): # Gensim 4.x Word2Vec/FastText format
        valid_words = [word for word in tokens if word in model.wv]
        if not valid_words:
            return np.zeros(vector_size)
        return np.mean([model.wv[word] for word in valid_words], axis=0)
    else: # Gensim downloaded model format (GloVe)
        valid_words = [word for word in tokens if word in model]
        if not valid_words:
            return np.zeros(vector_size)
        return np.mean([model[word] for word in valid_words], axis=0)

# Dictionary to hold our models for easy iteration
embedding_models = {
    "Word2Vec": w2v_model,
    "FastText": ft_model,
    "GloVe": glove_model
}

print("\n--- Model Comparison Results ---")

# Loop through each model to evaluate and compare
for model_name, current_model in embedding_models.items():
    print(f"\n--- Evaluating SVM with {model_name} Embeddings ---")
    
    # Generate vectors for the current model
    X_train_emb = np.array([get_doc_vector(tokens, current_model) for tokens in train_tokens])
    X_test_emb = np.array([get_doc_vector(tokens, current_model) for tokens in test_tokens])
    
    # 3. Train Classifier
    svm_classifier = SVC(kernel='linear')
    svm_classifier.fit(X_train_emb, y_train)
    
    # 4. Evaluate Model
    y_pred = svm_classifier.predict(X_test_emb)
    
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


--- Lab 2: Word Embeddings & Classification ---
Training Word2Vec model...
Training FastText model...
Loading pre-trained GloVe model (this downloads ~100MB on first run)...

--- Model Comparison Results ---

--- Evaluating SVM with Word2Vec Embeddings ---
Accuracy:  0.6694
Precision: 0.5777
Recall:    0.6694
F1-Score:  0.6103
Confusion Matrix:
 [[35  0  3  0  3]
 [ 2  0  1 41  2]
 [ 8  1 49  1  2]
 [ 0  0  1 48  5]
 [ 3  0  2  6 32]]

--- Evaluating SVM with FastText Embeddings ---
Accuracy:  0.3755
Precision: 0.3588
Recall:    0.3755
F1-Score:  0.3170
Confusion Matrix:
 [[41  0  0  0  0]
 [29  0  1 16  0]
 [28  0 33  0  0]
 [34  0  2 18  0]
 [36  0  2  5  0]]

--- Evaluating SVM with GloVe Embeddings ---
Accuracy:  0.9388
Precision: 0.9417
Recall:    0.9388
F1-Score:  0.9391
Confusion Matrix:
 [[39  0  1  0  1]
 [ 1 44  1  0  0]
 [ 5  1 54  0  1]
 [ 0  0  0 54  0]
 [ 2  0  1  1 39]]


In [17]:
# ==========================================
# SAVING & RELOADING MODELS
# ==========================================
print("\n--- Saving & Loading Pipeline ---")

# Save Models
joblib.dump(svm_classifier, 'svm_w2v_classifier.pkl')
w2v_model.save('w2v_news_model.bin')
print("Models saved successfully to disk.")

# Reload Models
loaded_svm = joblib.load('svm_w2v_classifier.pkl')
loaded_w2v = Word2Vec.load('w2v_news_model.bin')

# Classify new unseen article
new_article = "Apple has announced a new processor for its latest laptops, promising double the battery life and faster computing."
clean_new = preprocess_text(new_article).split()

# Reshape because Sklearn expects a 2D array for a single sample
new_vec = get_doc_vector(clean_new, loaded_w2v).reshape(1, -1)
predicted_category = loaded_svm.predict(new_vec)

print(f"\nUnseen Article: '{new_article}'")
print(f"Predicted Category: [{predicted_category[0].upper()}]")


--- Saving & Loading Pipeline ---
Models saved successfully to disk.

Unseen Article: 'Apple has announced a new processor for its latest laptops, promising double the battery life and faster computing.'
Predicted Category: [BUSINESS]
